# MNIST from scratch — exploration

This notebook is the *refactored* successor to the original
`Handwritten digits classification/Handwritten_digits_classifier (1).ipynb`
(kept untouched for history). Instead of redefining the network inline, it
**imports the same code the app and CLI use** from `src/`, so there is a
single source of truth.

Run from the repo root so the `src` package is importable.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # make `src` importable from notebooks/

import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import load_mnist
from src.network import NeuralNetwork

## 1. Load MNIST
`load_mnist()` downloads the raw IDX files on first run and returns
column-major arrays normalised to `[0, 1]`.

In [ ]:
X_train, Y_train, X_test, Y_test = load_mnist()
print('train', X_train.shape, 'test', X_test.shape)

# Peek at one image.
plt.imshow(X_train[:, 0].reshape(28, 28), cmap='Blues')
plt.title(f'label = {Y_train[0]}')
plt.axis('off');

## 2. Train the from-scratch network
`784 -> 128 -> 64 -> 10`, ReLU hidden layers, softmax + cross-entropy loss,
mini-batch gradient descent. This reaches ~98% test accuracy in ~15s.

In [ ]:
net = NeuralNetwork([784, 128, 64, 10], loss='cross_entropy', seed=42)
history = net.train(X_train, Y_train, epochs=30, lr=0.5, batch_size=64,
                    X_val=X_test, Y_val=Y_test, log_every=5)

In [ ]:
test_acc = net.accuracy(net.predict(X_test), Y_test)
print(f'test accuracy: {test_acc*100:.2f}%')

## 3. Loss / accuracy curve

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(history['epoch'], history['loss'], 'r-', label='loss')
ax1.set_xlabel('epoch'); ax1.set_ylabel('loss', color='r')
ax2 = ax1.twinx()
ax2.plot(history['epoch'], [a*100 for a in history['train_acc']], 'b-', label='train acc')
ax2.plot(history['epoch'], [a*100 for a in history['val_acc']], 'g--', label='test acc')
ax2.set_ylabel('accuracy (%)', color='b')
plt.title('Training curve'); plt.show()

## 4. Compare cross-entropy vs MSE (Step 3)
Same architecture and hyper-parameters, only the loss changes.

In [ ]:
for loss_name in ['cross_entropy', 'mse']:
    n = NeuralNetwork([784, 128, 64, 10], loss=loss_name, seed=42)
    n.train(X_train, Y_train, epochs=30, lr=0.5, batch_size=64, verbose=False)
    acc = n.accuracy(n.predict(X_test), Y_test)
    print(f'{loss_name:>14}: test acc {acc*100:.2f}%')

## 5. Inspect a prediction

In [ ]:
i = 98
proba = net.predict_proba(X_test[:, i:i+1]).ravel()
print('predicted', proba.argmax(), '| actual', Y_test[i])
plt.imshow(X_test[:, i].reshape(28, 28), cmap='Blues'); plt.axis('off');